# Set-up

In [ ]:
fold_id = 1
k = 5

frac = 0.8

random_seed = 0 # 69, 88

import time
start_time = str(time.ctime()).replace(':','-').replace(' ','_')

batch_size = 100
epochs = 100

p_dropout= 0.2
fingerprint_dim = 200

weight_decay = 5 # also known as l2_regularization_lambda
learning_rate = 2
output_units_num = 1 # for regression model
radius = 2
T = 4

task_name = 'lgFu'
tag = 'CT_train_test'


In [ ]:
# %%
import sys
import os

# %%
import os
import torch
import torch.autograd as autograd
import torch.nn as nn
import torch.nn.functional as F

import torch.optim as optim
import torch.utils.data as Data
torch.manual_seed(8) # for reproduce


import numpy as np
import gc
import sys
sys.setrecursionlimit(50000)
import pickle
torch.backends.cudnn.benchmark = True
torch.set_default_tensor_type('torch.cuda.FloatTensor')
# from tensorboardX import SummaryWriter
torch.nn.Module.dump_patches = True
import copy
import pandas as pd
#then import my own modules
from AttentiveFP import Fingerprint, Fingerprint_viz, save_smiles_dicts, get_smiles_dicts, get_smiles_array, moltosvg_highlight

# %%
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import QED
from rdkit.Chem import rdMolDescriptors, MolSurf
from rdkit.Chem.Draw import SimilarityMaps
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D

from numpy.polynomial.polynomial import polyfit
import matplotlib.pyplot as plt
from matplotlib import gridspec
import matplotlib.cm as cm
import matplotlib
%matplotlib inline

import seaborn as sns; sns.set_style("darkgrid")
from IPython.display import SVG, display
#import sascorer
import itertools
from sklearn.metrics import r2_score, mean_squared_error
import scipy

# Task data

In [ ]:
import os

#the key dir variable we need
#split_dir = f'{splitter_name.replace("Splitter", "")}'


train_dir = f'{task_name}/cv_{k}_fold'
fold_dir = f'{train_dir}/fold_{fold_id}'

mod_dir = f'{fold_dir}/saved_{task_name}_mods'
res_dir = os.path.join(fold_dir, 'results')

emb_dir = os.path.join(fold_dir, 'embeddings')

os.makedirs(mod_dir, exist_ok=True)
os.makedirs(res_dir, exist_ok=True)
os.makedirs(emb_dir, exist_ok=True)

In [ ]:
raw_filename = 'data_prep/training_set_with_CT_train_test.csv'
df_all = pd.read_csv(f'{raw_filename}', sep=',')

# Attentive FP data process

In [ ]:
raw_filename

In [ ]:
tasks = [task_name]

feature_filename = raw_filename.replace('.csv','.pickle')
filename = raw_filename.replace('.csv','')
prefix_filename = raw_filename.split('/')[-1].replace('.csv','')
smiles_tasks_df = pd.read_csv(raw_filename,sep=',')
smiles_tasks_df=smiles_tasks_df.rename(columns={'SMILES':'Smiles'})
smilesList = smiles_tasks_df.Smiles.values
print("number of all smiles: ",len(smilesList))
atom_num_dist = []
remained_smiles = []
canonical_smiles_list = []
for smiles in smilesList:
    try:        
        mol = Chem.MolFromSmiles(smiles)
        atom_num_dist.append(len(mol.GetAtoms()))
        remained_smiles.append(smiles)
        canonical_smiles_list.append(Chem.MolToSmiles(Chem.MolFromSmiles(smiles), isomericSmiles=True))
    except:
        print(smiles)
        pass
print("number of successfully processed smiles: ", len(remained_smiles))
smiles_tasks_df = smiles_tasks_df[smiles_tasks_df["Smiles"].isin(remained_smiles)]
# print(smiles_tasks_df)
smiles_tasks_df['cano_smiles'] =canonical_smiles_list
#assert canonical_smiles_list[8]==Chem.MolToSmiles(Chem.MolFromSmiles(smiles_tasks_df['cano_smiles'][8]), isomericSmiles=True)


In [ ]:
if os.path.isfile(feature_filename):
    feature_dicts = pickle.load(open(feature_filename, "rb" ))
else:
    feature_dicts = save_smiles_dicts(smilesList,filename)
# feature_dicts = get_smiles_dicts(smilesList)
remained_df = smiles_tasks_df[smiles_tasks_df["cano_smiles"].isin(feature_dicts['smiles_to_atom_mask'].keys())]
uncovered_df = smiles_tasks_df.drop(remained_df.index)
print(remained_df.columns)
print(remained_df.shape)

# df_cl_all.pickle is the feature dictionary

In [ ]:
remained_df.columns

# Data Splitting

In [ ]:

from sklearn.model_selection import train_test_split, KFold

def train_val_split_method(df, frac_train, random_seed, column_names):
    # Perform train-validation split
    train_df, valid_df = train_test_split(df, train_size=frac_train, random_state=random_seed)
    
    # Set column names
    train_df.columns = column_names
    valid_df.columns = column_names
    
    return train_df, valid_df

def k_fold_split_method(df, k, column_names):
    # Perform k-fold split
    kf = KFold(n_splits=k, shuffle=True, random_state=42)
    
    train_datasets = {}
    cv_datasets = {}
    
    for i, (train_index, cv_index) in enumerate(kf.split(df)):
        train_df = df.iloc[train_index].copy()
        cv_df = df.iloc[cv_index].copy()
        
        # Set column names
        train_df = train_df[column_names].copy()
        cv_df = cv_df[column_names].copy()
        
        train_datasets[f"train_fold_{i+1}"] = train_df
        cv_datasets[f"cv_fold_{i+1}"] = cv_df
    
    return train_datasets, cv_datasets


In [ ]:
training_all = remained_df.loc[remained_df[tag] == 'train'].copy()
test_df = remained_df.loc[remained_df[tag] == 'test'].copy()

# training_all = remained_df.loc[remained_df['tag'] == 'train' ].dropna(subset=f'{task_name}')
# test_df = remained_df[remained_df['tag'] == 'test'].copy()

In [ ]:

# Train-validation split
# train_df, valid_df = train_val_split_method(remained_df, frac_train=0.8, random_seed=42, column_names=column_names)

# K-fold split
train_datasets, cv_datasets = k_fold_split_method(training_all, 
                                                  k=k, 
                                                  column_names = ['PUBCHEM_CID',task_name, 'cano_smiles'] )

In [ ]:
train_df = train_datasets[f'train_fold_{fold_id}'].copy().reset_index(drop=True)
valid_df = cv_datasets[f'cv_fold_{fold_id}'].copy().reset_index(drop=True) 


In [ ]:
print(f"{task_name},{k}fcv: the cv_fold_{fold_id} data shape:",
      '\n\t train', train_df.shape[0],
    '\n\t valid', valid_df.shape[0],
    '\n\t test',  test_df.shape[0] )

# Training

In [ ]:
torch.cuda.empty_cache()

In [ ]:
x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, smiles_to_rdkit_list = get_smiles_array([canonical_smiles_list[0]],feature_dicts)
num_atom_features = x_atom.shape[-1]
num_bond_features = x_bonds.shape[-1]
loss_function = nn.MSELoss()
model = Fingerprint(radius, T, num_atom_features, num_bond_features,
            fingerprint_dim, output_units_num, p_dropout)
model.cuda()

# optimizer = optim.Adam(model.parameters(), learning_rate, weight_decay=weight_decay)
optimizer = optim.Adam(model.parameters(), 10**-learning_rate, weight_decay=10**-weight_decay)
# optimizer = optim.SGD(model.parameters(), 10**-learning_rate, weight_decay=10**-weight_decay)

# tensorboard = SummaryWriter(log_dir="runs/"+start_time+"_"+prefix_filename+"_"+str(fingerprint_dim)+"_"+str(p_dropout))

model_parameters = filter(lambda p: p.requires_grad, model.parameters())
params = sum([np.prod(p.size()) for p in model_parameters])
print(params)
for name, param in model.named_parameters():
    if param.requires_grad:
        print(name, param.data.shape)

In [ ]:
def train(model, dataset, optimizer, loss_function):
    model.train()
    np.random.seed(epoch)
    valList = np.arange(0,dataset.shape[0])
    #shuffle them
    np.random.shuffle(valList)
    batch_list = []
    for i in range(0, dataset.shape[0], batch_size):
        batch = valList[i:i+batch_size]
        batch_list.append(batch)   
    for counter, batch in enumerate(batch_list):
        batch_df = dataset.iloc[batch,:]
        smiles_list = batch_df.cano_smiles.values
        y_val = batch_df[tasks[0]].values
        
        x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, smiles_to_rdkit_list = get_smiles_array(smiles_list,feature_dicts)
        atoms_prediction, mol_prediction = model(torch.Tensor(x_atom),torch.Tensor(x_bonds),torch.cuda.LongTensor(x_atom_index),torch.cuda.LongTensor(x_bond_index),torch.Tensor(x_mask))
        
        optimizer.zero_grad()
        loss = loss_function(mol_prediction, torch.Tensor(y_val).view(-1,1))     
        loss.backward()
        optimizer.step()
        
def eval(model, dataset):
    model.eval()
    eval_MAE_list = []
    eval_MSE_list = []
    valList = np.arange(0,dataset.shape[0])
    batch_list = []
    for i in range(0, dataset.shape[0], batch_size):
        batch = valList[i:i+batch_size]
        batch_list.append(batch) 
        
    actuals = []
    predictions = []
    for counter, eval_batch in enumerate(batch_list):
        batch_df = dataset.iloc[eval_batch,:]
        smiles_list = batch_df.cano_smiles.values
#         print(batch_df)
        y_val = batch_df[tasks[0]].values
        
        x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, smiles_to_rdkit_list = get_smiles_array(smiles_list,feature_dicts)
        atoms_prediction, mol_prediction = model(torch.Tensor(x_atom),torch.Tensor(x_bonds),torch.cuda.LongTensor(x_atom_index),torch.cuda.LongTensor(x_bond_index),torch.Tensor(x_mask))
        MAE = F.l1_loss(mol_prediction, torch.Tensor(y_val).view(-1,1), reduction='none')        
        MSE = F.mse_loss(mol_prediction, torch.Tensor(y_val).view(-1,1), reduction='none')
#         print(x_mask[:2],atoms_prediction.shape, mol_prediction,MSE)
        
        eval_MAE_list.extend(MAE.data.squeeze().cpu().numpy())
        eval_MSE_list.extend(MSE.data.squeeze().cpu().numpy())

        actuals.extend(y_val)
        predictions.extend(mol_prediction.data.squeeze().cpu().numpy())
        
    actuals = np.array(actuals)
    predictions = np.array(predictions)
    R2 = r2_score(actuals, predictions)
    RMSE = np.sqrt(mean_squared_error(actuals, predictions))
        
    return np.array(eval_MAE_list).mean(), np.array(eval_MSE_list).mean(), R2, RMSE

In [ ]:
import logging

# Ensure the necessary variables are defined
epoch = 0  # Example value, replace with actual epoch variable if defined elsewhere

# Set up logging
logging.basicConfig(level=logging.INFO, 
                    format='%(asctime)s - %(levelname)s - %(message)s',
                    filename=f'{mod_dir}/{prefix_filename}_{start_time}_{epoch}.log', 
                    filemode='w')
console = logging.StreamHandler()

logging.getLogger('').addHandler(console)

In [ ]:
best_param = {}
best_param["train_epoch"] = 0
best_param["valid_epoch"] = 0
best_param["train_MSE"] = 9e8
best_param["valid_MSE"] = 9e8

train_MSE_history = []
valid_MSE_history = []
train_R2_history = []
valid_R2_history = []

for epoch in range(1000):
    train_MAE, train_MSE, train_R2, train_RMSE = eval(model, train_df)
    valid_MAE, valid_MSE, valid_R2, valid_RMSE = eval(model, valid_df)
    
    train_MSE_history.append(train_MSE)
    valid_MSE_history.append(valid_MSE)

    train_R2_history.append(train_R2)
    valid_R2_history.append(valid_R2)

     # Update best parameters for training 
    if train_MSE < best_param["train_MSE"]:
        best_param["train_epoch"] = epoch
        best_param["train_MSE"] = train_MSE
    # Update best parameters for validation and potentially save the model
    if valid_MSE < best_param["valid_MSE"]:
        best_param["valid_epoch"] = epoch
        best_param["valid_MSE"] = valid_MSE
        # Save the model if RMSE on validation is less than 0.60
        if np.sqrt(valid_MSE) < 0.65:
            torch.save(model, f"{mod_dir}/{prefix_filename}_{start_time}_{best_param['valid_epoch']}.pt")
    if (epoch - best_param["train_epoch"] > 8) and (epoch - best_param["valid_epoch"] > 18):
        break

    # Log the epoch results
    logging.info(f"Epoch: {epoch}, Train RMSE: {np.sqrt(train_MSE):.3f}, Valid RMSE: {np.sqrt(valid_MSE):.3f}, "
                 f"Train R2: {train_R2:.3f}, Valid R2: {valid_R2:.3f}")
    
    # Training step
    train(model, train_df, optimizer, loss_function)


In [ ]:
print("best valid-epoch is")
print(prefix_filename+'_'+start_time+'_'+str(best_param["valid_epoch"])+'.pt')

best_mod = f"{prefix_filename}_{start_time}_{best_param['valid_epoch']}.pt"

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot(train_R2_history, label='Train R2')
plt.plot(valid_R2_history, label='Valid R2')
plt.title(f'R2 score During Training with Learning rate:10^-{learning_rate}',fontsize = 12)
plt.xlabel('Epoch', fontsize =12)
plt.ylabel('Coefficient of Determination',fontsize =12)
#plt.yscale('log')
plt.ylim(top=1) 
plt.legend()
plt.grid(True)
# plt.show()
plt.savefig(f"{res_dir}/R2_{best_mod}.png", format = 'png', dpi = 600)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.plot(np.sqrt(train_MSE_history), label=f'Train RMSE')
plt.plot(np.sqrt(valid_MSE_history), label='Valid RMSE')
plt.title(f'RMSE During Training with Learning rate:10^-{learning_rate}',fontsize = 12)
plt.xlabel('Epoch', fontsize =12)
plt.ylabel('Root Mean Squared Error',fontsize =12)
#plt.yscale('log')
plt.ylim(top=1) 
plt.legend()
plt.grid(True)
# plt.show()
plt.savefig(f"{res_dir}/RMSE_{best_mod}.png", format = 'png', dpi = 600)

# Evaluation

In [ ]:
# best_mod = 'df_feature_5620_Tue_Apr__1_06-43-40_2025_37.pt'
# combined_results = f"{res_dir}/metrics_{best_mod}.csv"


In [ ]:
# evaluate model
model_eval = torch.load(f'{mod_dir}/{best_mod}')   
# best_model_dict = best_model.state_dict()
# best_model_wts = copy.deepcopy(best_model_dict)

# model.load_state_dict(best_model_wts)
# (best_model.align[0].weight == model.align[0].weight).all()

In [ ]:
def eval_new(model, dataset):
    model.eval()
    eval_MAE_list = []
    eval_MSE_list = []
    valList = np.arange(0,dataset.shape[0])
    batch_list = []
    for i in range(0, dataset.shape[0], batch_size):
        batch = valList[i:i+batch_size]
        batch_list.append(batch) 
        
    actuals = []
    predictions = []
    for counter, eval_batch in enumerate(batch_list):
        batch_df = dataset.iloc[eval_batch,:]
        smiles_list = batch_df.cano_smiles.values
#         print(batch_df)
        y_val = batch_df[tasks[0]].values
        
        x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, smiles_to_rdkit_list = get_smiles_array(smiles_list,feature_dicts)
        atoms_prediction, mol_prediction = model(torch.Tensor(x_atom),torch.Tensor(x_bonds),torch.cuda.LongTensor(x_atom_index),torch.cuda.LongTensor(x_bond_index),torch.Tensor(x_mask))
        MAE = F.l1_loss(mol_prediction, torch.Tensor(y_val).view(-1,1), reduction='none')        
        MSE = F.mse_loss(mol_prediction, torch.Tensor(y_val).view(-1,1), reduction='none')
#         print(x_mask[:2],atoms_prediction.shape, mol_prediction,MSE)
        
        eval_MAE_list.extend(MAE.data.squeeze().cpu().numpy())
        eval_MSE_list.extend(MSE.data.squeeze().cpu().numpy())

        actuals.extend(y_val)
        predictions.extend(mol_prediction.data.squeeze().cpu().numpy())
        
    actuals = np.array(actuals)
    predictions = np.array(predictions)
    R2 = r2_score(actuals, predictions)
    RMSE = np.sqrt(mean_squared_error(actuals, predictions))
    
    return actuals, predictions

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from collections import Counter
import math

In [ ]:
def compute_acc(y_true, y_pred, threshold=0.1):
    """计算绝对误差小于阈值的预测的比例"""
    AE = np.abs(y_pred - y_true)  # 计算绝对误差
    num_within_threshold = np.sum(AE <= threshold)  # 计算满足阈值条件的预测数量
    total_predictions = len(y_pred)  # 总预测数量
    accuracy = num_within_threshold / total_predictions  # 计算精确度
    return accuracy
    
def compute_acc(y_true, y_pred, threshold=0.1):
    """计算绝对误差小于阈值的预测的比例"""
    AE = np.abs(y_pred - y_true)  # 计算绝对误差
    num_within_threshold = np.sum(AE <= threshold)  # 计算满足阈值条件的预测数量
    total_predictions = len(y_pred)  # 总预测数量
    accuracy = num_within_threshold / total_predictions  # 计算精确度
    return accuracy
    
def compute_gmfe(y_true, y_pred):
    """计算几何平均折叠误差（Geometric Mean Fold Error）"""
    epsilon = 1e-10  # 避免除以 0
    y_true = np.maximum(10 ** y_true, epsilon)
    y_pred = np.maximum(10 ** y_pred, epsilon)
    ratios = np.maximum(y_pred / y_true, y_true / y_pred)
    gmfe =  np.exp(np.mean(np.log(ratios)))

    # fold_errors = y_pred / y_true  # pred/true, if < 1, underpredict 
    # gmfe = np.exp(np.mean(np.log(fold_errors)))
    return gmfe


def compute_fe_freq(y_true, y_pred, threshold=2):
    """计算给定阈值下的折叠误差频率"""
    epsilon = 1e-10
    y_true = np.maximum(10 ** y_true, epsilon)
    y_pred = np.maximum(10 ** y_pred, epsilon)
    
    ratios = np.maximum(y_pred / y_true, y_true / y_pred)
    freq = np.mean(ratios <= threshold)
    return freq

def eval_plot(y_act, y_pred, plot_title, res_dir=res_dir):
    # Compute metrics
    metrics = {
        'MAE': mean_absolute_error(y_act, y_pred),
        'MSE': mean_squared_error(y_act, y_pred),
        'R2': r2_score(y_act, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_act, y_pred)),
        'ACC': compute_acc(np.array(y_act), np.array(y_pred)),  # Define compute_acc elsewhere
        'GMFE': compute_gmfe(np.array(y_act), np.array(y_pred)), 
        'FE<2': compute_fe_freq(np.array(y_act), np.array(y_pred), 2),  
        'FE<3': compute_fe_freq(np.array(y_act), np.array(y_pred), 3),   
        'FE<4': compute_fe_freq(np.array(y_act), np.array(y_pred), 4),
        'FE<5': compute_fe_freq(np.array(y_act), np.array(y_pred), 5)
    }

#     # Display results
#     print(f"{plot_title}:")
#     for key, value in metrics.items():
#         if key.startswith('FE_FREQ'):
#             print(f"\t % {key}: {value[2]*100:.2f}%")
#         else:
#             print(f"\t {key}: {value:.2f}")


#  # Plotting
#     fig, ax = plt.subplots(figsize=(5, 5))
#     ax.scatter(y_act, y_pred, edgecolors='k', facecolors='none', label=f"Predicted vs. Actual\n$R^2={metrics['R2']:.2f}$")
#     ax.plot([min(y_act), max(y_act)], [min(y_act), max(y_act)], 'k--', lw=2, label='Perfect Prediction (y=x)')
#     error_margin = 1  # Error margin
#     ax.plot([min(y_act), max(y_act)], [min(y_act) - error_margin, max(y_act) - error_margin], 'r--', lw=1.5, label='Lower Error Bound')
#     ax.plot([min(y_act), max(y_act)], [min(y_act) + error_margin, max(y_act) + error_margin], 'b--', lw=1.5, label='Upper Error Bound')
    
#     ax.set_xlabel('Measured logCL', fontsize=12)
#     ax.set_ylabel('Predicted logCL', fontsize=12)
#     ax.set_title(plot_title, fontsize=14)
#     ax.legend(facecolor='white', framealpha=1)
#     ax.grid(True)
    
#     # Save the plot
#     filename = f'{res_dir}/{plot_title}.png'
#     plt.savefig(filename, format='png', dpi=600)
#     plt.show()
#     # Return results as a DataFrame for a single evaluation

    return pd.DataFrame(metrics, index=[0]) 

def compute_fe_freq(y_true, y_pred, threshold=2):
    """计算给定阈值下的折叠误差频率基于原始值"""
    # 先将对数值转换回原始数值
    
    original_y_pred = 10 ** y_pred
    original_y_true = 10 ** y_true
    
    # 计算两者间的比率
    ratios = np.maximum(original_y_pred / original_y_true, original_y_true / original_y_pred)
    
    # 计算小于等于各阈值的比率的频率
    freq = np.mean(ratios <= threshold)
    return freq


def eval_plot(y_act, y_pred, plot_title, res_dir=res_dir):
    # Compute metrics
    metrics = {
        'MAE': mean_absolute_error(y_act, y_pred),
        'MSE': mean_squared_error(y_act, y_pred),
        'R2': r2_score(y_act, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_act, y_pred)),
        'ACC': compute_acc(np.array(y_act), np.array(y_pred)),  # Define compute_acc elsewhere
        'GMFE': compute_gmfe(np.array(y_act), np.array(y_pred)), 
        'FE<2': compute_fe_freq(np.array(y_act), np.array(y_pred), 2),  
        'FE<3': compute_fe_freq(np.array(y_act), np.array(y_pred), 3),   
        'FE<4': compute_fe_freq(np.array(y_act), np.array(y_pred), 4),
        'FE<5': compute_fe_freq(np.array(y_act), np.array(y_pred), 5)
    }

#     # Display results
#     print(f"{plot_title}:")
#     for key, value in metrics.items():
#         if key.startswith('FE_FREQ'):
#             print(f"\t % {key}: {value[2]*100:.2f}%")
#         else:
#             print(f"\t {key}: {value:.2f}")


#  # Plotting
#     fig, ax = plt.subplots(figsize=(5, 5))
#     ax.scatter(y_act, y_pred, edgecolors='k', facecolors='none', label=f"Predicted vs. Actual\n$R^2={metrics['R2']:.2f}$")
#     ax.plot([min(y_act), max(y_act)], [min(y_act), max(y_act)], 'k--', lw=2, label='Perfect Prediction (y=x)')
#     error_margin = 1  # Error margin
#     ax.plot([min(y_act), max(y_act)], [min(y_act) - error_margin, max(y_act) - error_margin], 'r--', lw=1.5, label='Lower Error Bound')
#     ax.plot([min(y_act), max(y_act)], [min(y_act) + error_margin, max(y_act) + error_margin], 'b--', lw=1.5, label='Upper Error Bound')
    
#     ax.set_xlabel('Measured logCL', fontsize=12)
#     ax.set_ylabel('Predicted logCL', fontsize=12)
#     ax.set_title(plot_title, fontsize=14)
#     ax.legend(facecolor='white', framealpha=1)
#     ax.grid(True)
    
#     # Save the plot
#     filename = f'{res_dir}/{plot_title}.png'
#     plt.savefig(filename, format='png', dpi=600)
#     plt.show()
#     # Return results as a DataFrame for a single evaluation

    return pd.DataFrame(metrics, index=[0]) 

In [ ]:
def prepare_and_evaluate_all_data(model, datasets, eval_function):
    # Initialize an empty list to store DataFrame segments
    results = []

    # Loop through each dataset entry which is a tuple of (dataset_df, dataset_type)
    for dataset_df, dataset_type in datasets:
        # Evaluate the model on the provided dataset
        actuals, preds = eval_function(model, dataset_df)
        
        # Create a DataFrame with the results and the dataset type
        df = pd.DataFrame({
            'Actual': actuals,
            'Predicted': preds,
            'Dataset': [dataset_type] * len(actuals)
        })
        
        # Append the DataFrame to the list
        results.append(df)

    # Concatenate all DataFrame segments into one DataFrame
    combined_results = pd.concat(results, ignore_index=True)
    return combined_results

def apply_eval_plot_by_dataset_type(dataframe, res_dir):
    dataset_types = dataframe['Dataset'].unique()  # 获取所有唯一的数据集类型

    res_metrics = []

    for dataset_type in dataset_types:
        # 选择当前类型的数据
        subset = dataframe[dataframe['Dataset'] == dataset_type]
        actuals = subset['Actual'].values
        preds = subset['Predicted'].values
        
        metrics_df = eval_plot(actuals, preds, f"AttentiveFP_{dataset_type.lower()}_{task_name}", res_dir)
        metrics_df['Dataset'] = dataset_type
        res_metrics.append(metrics_df)

    # 使用 pd.concat 来合并所有 DataFrame，假设它们具有相同的列结构
    combined_metrics_df = pd.concat(res_metrics, ignore_index=True)

    return combined_metrics_df


In [ ]:
data_eval = [
    (train_df, f'{task_name}_train'),
    (valid_df, f'{task_name}_valid'),
    (test_df, f'{task_name}_test')
]

combined_results = prepare_and_evaluate_all_data(model = model_eval,
                                                  datasets = data_eval, 
                                                  eval_function = eval_new)

In [ ]:
print(combined_results.head())

In [ ]:
combined_metrics = apply_eval_plot_by_dataset_type(combined_results, res_dir)


In [ ]:
# combined_metrics['Epoch']=best_param["valid_epoch"]
combined_metrics['mod_name']=best_mod

combined_metrics = combined_metrics[ ['Dataset']+[col for col in combined_metrics.columns if col  != 'Dataset' ] ]
print( combined_metrics)

In [ ]:
combined_metrics.to_csv(f"{res_dir}/metrics_{best_mod}.csv", index = False)
combined_results.to_csv(f"{res_dir}/results_{best_mod}.csv", index = False)


In [ ]:
print(best_mod)

# Visualization

In [ ]:
sns.lmplot(data=combined_results, 
           x= 'Actual',
           y= 'Predicted',
           hue = 'Dataset',
           markers = ['o', 'x', '+'],
           palette = 'deep')
plt.show()

In [ ]:
g = sns.FacetGrid(combined_results, col='Dataset', height=3, aspect=1)

# 显式地调用绘图函数和数据
def plot_func(data, **kwargs):
    ax = plt.gca()
    r2 = r2_score(data['Actual'], data['Predicted'])
    sns.regplot(data=data, x='Actual', y='Predicted', **kwargs)
    
    ax.text(0.05, 0.95, f'R²: {r2:.2f}', transform=ax.transAxes, verticalalignment='top')

g.map_dataframe(plot_func)

                 
plt.show()

# Meta-Features Generator

In [ ]:
torch.cuda.empty_cache()


import torch
import numpy as np
import random
import pandas as pd

def setup_seed(seed=0):
    """ Set seed for reproducibility. """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # For multi-GPU setups, if any.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False  # May impact performance negatively if network input size varies
    torch.set_num_threads(1)  # Can limit the number of threads to reduce randomness in multithreading.



def generate_embeddings(model, dataset, batch_size, feature_dicts):
    # Calculate the number of batches
    n_batches = len(dataset) // batch_size + (0 if len(dataset) % batch_size == 0 else 1)
    
    # Initialize an empty DataFrame to store features
    df_features = pd.DataFrame()
    
    for i in range(n_batches):
        # Select the batch
        batch_df = dataset.iloc[i*batch_size:(i+1)*batch_size, :]
        smiles_list = batch_df.cano_smiles.values
        
        # Extract features for the batch
        x_atom, x_bonds, x_atom_index, x_bond_index, x_mask, smiles_to_rdkit_list = get_smiles_array(smiles_list, feature_dicts)
        
        # Model prediction
        _, mol_prediction = model(
            torch.Tensor(x_atom),
            torch.Tensor(x_bonds),
            torch.cuda.LongTensor(x_atom_index),
            torch.cuda.LongTensor(x_bond_index),
            torch.Tensor(x_mask)
        )

        # Create DataFrame for the current batch
        batch_features = pd.DataFrame(
            data=mol_prediction.cpu().data.numpy(),
            index=batch_df.iloc[:,0]
        )
        
        
        # Append the current batch's features to the main DataFrame
        df_features = pd.concat([df_features, batch_features])
    
    return df_features


In [ ]:
setup_seed()
import torch
import torch.nn as nn

def feature_save(mod_path, num_embeddings, dataset, task):

    # setting-up
    feture_dicts = {}
    
    # Load the models
    model = torch.load(mod_path)
    # Update the model's output layer
    model.output = nn.Sequential(nn.Linear(in_features=fingerprint_dim, out_features=num_embeddings, bias=True))

    # Generate features
    df_features = generate_embeddings(model, dataset, batch_size, feature_dicts)

    # Save the features to a CSV file
    new_column_names = [f"{task}_{i}" for i in range(1, df_features.shape[1]+1)]

    df_features.columns = new_column_names

    df_features.reset_index(inplace=True)   

    ### saving 

    csv_filename = f'emb_dir/{task}_embs_{num_embeddings}.csv'
    df_features.to_csv(csv_filename, index=False)
    
    print(f"Features are saved to '{csv_filename}' and the shape of the DataFrame is:", df_features.shape)
    print("\n", df_features)
    
    return df_features


In [ ]:
best_mod = 'df_feature_5620_Sat_Apr__5_21-26-10_2025_83.pt'


for n_emb in [10,25,50,100,200]:
    df_emb = feature_save(mod_path = f'{mod_dir}/{best_mod}', 
                            num_embeddings=n_emb, 
                            dataset = remained_df,
                            task = task_name.replace('lg', ''))